# Deep Learning 基礎講座　最終課題: 脳波分類

## 概要
被験者が画像を見ているときの脳波から，その画像がどのカテゴリに属するかを分類するタスク．
- サンプル数: 訓練 118,800 サンプル，検証 59,400 サンプル，テスト 59,400 サンプル
- クラス数: 5
- 入力: 脳波データ（チャンネル数 x 系列長）
- 出力: 対応する画像のクラス
- 評価指標: Top-1 accuracy

### 元データセット ([Gifford2022 EEG dataset](https://osf.io/3jk45/)) との違い

- 本コンペでは難易度調整の目的で元データセットにいくつかの改変を加えています．

1. 訓練セットのみの使用
  - 元データセットでは訓練データに存在しなかったクラスの画像を見ているときの脳波においてテストが行われますが，これは難易度が非常に高くなります．
  - 本コンペでは元データセットの訓練セットを再分割し，訓練時に存在した画像に対応する別の脳波において検証・テストを行います．

2. クラス数の減少
  - 元データセット（の訓練セット）では16,540枚の画像に対し，1,654のクラスが存在します．
    - e.g. `aardvark`, `alligator`, `almond`, ...
  - 本コンペでは1,654のクラスを，`animal`, `food`, `clothing`, `tool`, `vehicle`の5つにまとめています．
    - e.g. `aardvark -> animal`, `alligator -> animal`, `almond -> food`, ...

### 考えられる工夫の例

- 音声モデルの導入
  - 脳波と同じ波である音声を扱うアーキテクチャを用いることが有効であると知られています．
  - 例）Conformer [[Gulati+ 2020](https://arxiv.org/abs/2005.08100)]
- 画像データを用いた事前学習
  - 本コンペのタスクは脳波のクラス分類ですが，配布してある画像データを脳波エンコーダの事前学習に用いることを許可します．
  - 例）CLIP [Radford+ 2021]
  - 画像を用いる場合は[こちら](https://osf.io/download/3v527/)からダウンロードしてください．
- 過学習を防ぐ正則化やドロップアウト


## 修了要件を満たす条件
- ベースラインモデルのbest test accuracyは38.7%となります．**これを超えた提出のみ，修了要件として認めます**．
- ベースラインから改善を加えることで，55%までは性能向上することを運営で確認しています．こちらを 1 つの指標として取り組んでみてください．

## 注意点
- 学習するモデルについて制限はありませんが，必ず訓練データで学習したモデルで予測してください．
    - 事前学習済みモデルを利用して，訓練データを fine-tuning しても構いません．
    - 埋め込み抽出モデルなど，モデルの一部を訓練しないケースは構いません．
    - 学習を一切せずに，ChatGPT などの基盤モデルを利用することは禁止とします．

## 1.準備

In [ ]:
DETAILED_MAPPING = {
    'aardvark': 'animal',
    'abacus': 'tool',
    'accordion': 'tool',
    'acorn': 'food',
    'air_conditioner': 'tool',
    'air_mattress': 'tool',
    'air_pump': 'tool',
    'airbag': 'tool',
    'airboat': 'vehicle',
    'airplane': 'vehicle',
    'album': 'tool',
    'alligator': 'animal',
    'almond': 'food',
    'aloe': 'tool',
    'alpaca': 'animal',
    'altar': 'tool',
    'aluminum_foil': 'tool',
    'amber': 'tool',
    'ambulance': 'vehicle',
    'amplifier': 'tool',
    'anchor': 'tool',
    'ankle': 'tool',
    'anklet': 'tool',
    'ant': 'animal',
    'anteater': 'animal',
    'antenna': 'tool',
    'anvil': 'tool',
    'appetizer': 'food',
    'apple': 'food',
    'apple_tree': 'food',
    'applesauce': 'food',
    'apron': 'clothing',
    'aquarium': 'tool',
    'arch': 'tool',
    'arm': 'tool',
    'armor': 'tool',
    'arrow': 'tool',
    'artichoke': 'food',
    'arugula': 'food',
    'ashtray': 'tool',
    'asparagus': 'food',
    'avocado': 'food',
    'awning': 'tool',
    'axe': 'tool',
    'baby': 'tool',
    'backdrop': 'tool',
    'backgammon': 'tool',
    'backpack': 'tool',
    'bacon': 'food',
    'badge': 'tool',
    'badger': 'animal',
    'bag': 'tool',
    'bagel': 'food',
    'bagpipe': 'tool',
    'baklava': 'food',
    'ball': 'tool',
    'balloon': 'tool',
    'ballot_box': 'tool',
    'bamboo': 'tool',
    'banana_peel': 'food',
    'banana_split': 'food',
    'bandage': 'tool',
    'bandanna': 'clothing',
    'banjo': 'tool',
    'bank': 'tool',
    'banner': 'tool',
    'barbed_wire': 'tool',
    'barbell': 'tool',
    'barcode': 'tool',
    'bark': 'tool',
    'barnacle': 'animal',
    'barrel': 'tool',
    'barrette': 'tool',
    'baseball': 'tool',
    'baseball_glove': 'clothing',
    'basket': 'tool',
    'basketball_hoop': 'tool',
    'bassinet': 'tool',
    'bat1': 'tool',
    'bat2': 'animal',
    'bathmat': 'tool',
    'bathrobe': 'clothing',
    'bathtub': 'tool',
    'baton1': 'tool',
    'baton2': 'tool',
    'baton3': 'tool',
    'battery': 'tool',
    'bazooka': 'tool',
    'beachball': 'tool',
    'bead': 'tool',
    'beaker': 'tool',
    'bean': 'food',
    'beanbag': 'tool',
    'beanie': 'clothing',
    'bear': 'animal',
    'beard': 'tool',
    'bed': 'tool',
    'bedpan': 'tool',
    'bedpost': 'tool',
    'bee': 'animal',
    'beehive': 'animal',
    'beer': 'food',
    'beet': 'food',
    'beetle': 'animal',
    'bell': 'tool',
    'bell_pepper': 'food',
    'belt': 'clothing',
    'belt_buckle': 'clothing',
    'berry': 'food',
    'bib': 'clothing',
    'bikini': 'clothing',
    'bin': 'tool',
    'binder': 'tool',
    'binoculars': 'tool',
    'bird': 'animal',
    'birdbath': 'tool',
    'birdcage': 'tool',
    'birdhouse': 'tool',
    'biscuit': 'food',
    'bison': 'animal',
    'blackberry': 'food',
    'blanket': 'tool',
    'blazer': 'clothing',
    'blender': 'tool',
    'blimp': 'vehicle',
    'blind': 'tool',
    'blinder': 'tool',
    'blindfold': 'tool',
    'block': 'tool',
    'blouse': 'clothing',
    'blower': 'tool',
    'blowfish': 'animal',
    'blowgun': 'tool',
    'blueberry': 'food',
    'boa': 'animal',
    'boar': 'animal',
    'board': 'tool',
    'board_game': 'tool',
    'bobsled': 'vehicle',
    'bolo_tie': 'clothing',
    'bologna': 'food',
    'bolt': 'tool',
    'bomb': 'tool',
    'bone': 'tool',
    'bongo': 'tool',
    'bonsai': 'tool',
    'book': 'tool',
    'bookmark': 'tool',
    'bookshelf': 'tool',
    'boomerang': 'tool',
    'boot': 'clothing',
    'bottle': 'tool',
    'boulder': 'tool',
    'bouquet': 'tool',
    'bow1': 'tool',
    'bow2': 'tool',
    'bow3': 'tool',
    'bowl': 'tool',
    'bowler_hat': 'clothing',
    'bowling_ball': 'tool',
    'bowtie': 'clothing',
    'box': 'tool',
    'boxer_shorts': 'clothing',
    'boxing_gloves': 'clothing',
    'boy': 'tool',
    'bra': 'clothing',
    'bracelet1': 'clothing',
    'bracelet2': 'clothing',
    'bracket': 'tool',
    'braid': 'tool',
    'brake': 'tool',
    'branch': 'tool',
    'brass_knuckles': 'tool',
    'breadstick': 'food',
    'breakfast': 'food',
    'breathalyzer': 'tool',
    'brick': 'tool',
    'briefcase': 'tool',
    'broccoli': 'food',
    'brooch': 'clothing',
    'broom': 'tool',
    'brownie': 'food',
    'brush': 'tool',
    'brussels_sprouts': 'food',
    'bubble': 'tool',
    'bubble_wrap': 'tool',
    'bucket': 'tool',
    'buckle': 'clothing',
    'buffet': 'tool',
    'bull': 'animal',
    'bulldozer': 'vehicle',
    'bulletin_board': 'tool',
    'bulletproof_vest': 'clothing',
    'bumper': 'tool',
    'bungee': 'tool',
    'bunkbed': 'tool',
    'buoy': 'tool',
    'burner': 'tool',
    'burrito': 'food',
    'bus': 'vehicle',
    'butter': 'food',
    'butterfly': 'animal',
    'button1': 'tool',
    'button2': 'tool',
    'cabbage': 'food',
    'cabinet': 'tool',
    'cable': 'tool',
    'cactus': 'tool',
    'cage': 'tool',
    'cake': 'food',
    'cake_mix': 'food',
    'calculator': 'tool',
    'calf1': 'tool',
    'calf2': 'tool',
    'calzone': 'food',
    'camcorder': 'tool',
    'camel': 'animal',
    'camera1': 'tool',
    'camera2': 'tool',
    'camera_lens': 'tool',
    'camper': 'vehicle',
    'can': 'tool',
    'can_opener': 'tool',
    'candelabra': 'tool',
    'candle': 'tool',
    'candy': 'food',
    'candy_bar': 'food',
    'candy_cane': 'food',
    'cane': 'tool',
    'canister': 'tool',
    'cannon': 'tool',
    'cannonball': 'tool',
    'canoe': 'vehicle',
    'cantaloupe': 'food',
    'canvas': 'tool',
    'cap': 'clothing',
    'cape': 'clothing',
    'car': 'vehicle',
    'car_door': 'tool',
    'car_seat': 'tool',
    'caramel': 'food',
    'card': 'tool',
    'cardboard': 'tool',
    'cardigan': 'clothing',
    'cardinal': 'animal',
    'carousel': 'tool',
    'carriage': 'vehicle',
    'carrot': 'food',
    'cash_machine': 'tool',
    'cash_register': 'tool',
    'casserole': 'food',
    'cassette': 'tool',
    'catapult': 'tool',
    'catfish': 'animal',
    'cauliflower': 'food',
    'caviar': 'food',
    'celery': 'food',
    'cello': 'tool',
    'cellphone': 'tool',
    'cement_mixer': 'tool',
    'centerpiece': 'tool',
    'centrifuge': 'tool',
    'cereal': 'food',
    'chainsaw': 'tool',
    'chair': 'tool',
    'chalice': 'tool',
    'chalk': 'tool',
    'chalkboard': 'tool',
    'champagne': 'food',
    'chandelier': 'tool',
    'charcoal': 'tool',
    'charger': 'tool',
    'chariot': 'vehicle',
    'checkbook': 'tool',
    'checkers': 'tool',
    'cheeseburger': 'food',
    'cheesecake': 'food',
    'cherry': 'food',
    'chess_piece': 'tool',
    'chessboard': 'tool',
    'chest1': 'tool',
    'chick': 'animal',
    'chicken1': 'animal',
    'chicken2': 'animal',
    'chicken_wire': 'tool',
    'chickpea': 'food',
    'chihuahua': 'animal',
    'chili': 'food',
    'chimney': 'tool',
    'chin': 'tool',
    'chinaware': 'tool',
    'chinchilla': 'animal',
    'chip': 'tool',
    'chipmunk': 'animal',
    'chips': 'food',
    'chisel': 'tool',
    'chive': 'food',
    'chocolate': 'food',
    'christmas_card': 'tool',
    'christmas_tree': 'tool',
    'chute': 'tool',
    'cigar': 'tool',
    'cigarette': 'tool',
    'cigarette_butt': 'tool',
    'cigarette_holder': 'tool',
    'cilantro': 'food',
    'cinnamon': 'food',
    'clam': 'animal',
    'clarinet': 'tool',
    'clasp': 'tool',
    'clay': 'tool',
    'clipboard': 'tool',
    'clipper1': 'tool',
    'clipper2': 'tool',
    'cloak': 'clothing',
    'clock': 'tool',
    'closet': 'tool',
    'clothes': 'clothing',
    'clothesline': 'clothing',
    'clothespin': 'clothing',
    'cloud': 'tool',
    'clove': 'food',
    'clover': 'tool',
    'coal': 'tool',
    'coaster': 'tool',
    'coat_rack': 'clothing',
    'cockatoo': 'animal',
    'cockroach': 'animal',
    'cocktail': 'food',
    'cocoon': 'animal',
    'coffee': 'food',
    'coffee_filter': 'tool',
    'coffee_pot': 'tool',
    'coffee_table': 'tool',
    'coffin': 'tool',
    'coil': 'tool',
    'coin': 'tool',
    'coleslaw': 'food',
    'collar': 'clothing',
    'column': 'tool',
    'comb': 'tool',
    'combination_lock': 'tool',
    'comic_book': 'tool',
    'compass': 'tool',
    'compost': 'tool',
    'computer': 'tool',
    'computer_screen': 'tool',
    'confetti': 'tool',
    'contact_lens': 'tool',
    'container': 'tool',
    'cooker': 'tool',
    'cookie_cutter': 'food',
    'cookie_sheet': 'food',
    'cooler': 'tool',
    'coop': 'tool',
    'copier': 'tool',
    'coral': 'animal',
    'cord': 'tool',
    'cork': 'tool',
    'corkboard': 'tool',
    'corkscrew': 'tool',
    'corn': 'food',
    'cornbread': 'food',
    'cornhusk': 'food',
    'cornmeal': 'food',
    'cornucopia': 'food',
    'corsage': 'clothing',
    'corset': 'clothing',
    'costume': 'clothing',
    'cot': 'tool',
    'cotton_candy': 'food',
    'couch': 'tool',
    'cougar': 'animal',
    'counter': 'tool',
    'cow': 'animal',
    'coyote': 'animal',
    'cracker': 'food',
    'cranberry': 'food',
    'crane': 'vehicle',
    'crank': 'tool',
    'crate': 'tool',
    'crayfish': 'animal',
    'crayon': 'tool',
    'cream': 'food',
    'cream_cheese': 'food',
    'credit_card': 'tool',
    'cross': 'tool',
    'crossbow': 'tool',
    'crouton': 'food',
    'crowbar': 'tool',
    'crown': 'clothing',
    'crucifix': 'tool',
    'crutch': 'tool',
    'crystal1': 'tool',
    'crystal2': 'tool',
    'crystal_ball': 'tool',
    'cuckoo_clock': 'tool',
    'cucumber': 'food',
    'cufflink': 'clothing',
    'cummerbund': 'clothing',
    'cup': 'tool',
    'curb': 'tool',
    'curling_iron': 'tool',
    'curry': 'food',
    'curtain': 'tool',
    'cushion': 'tool',
    'cutting_board': 'tool',
    'cymbal': 'tool',
    'daisy': 'tool',
    'dandelion': 'tool',
    'dart': 'tool',
    'dartboard': 'tool',
    'dashboard': 'tool',
    'deer': 'animal',
    'defibrillator': 'tool',
    'denture': 'tool',
    'deodorant': 'tool',
    'desk': 'tool',
    'detonator': 'tool',
    'dial': 'tool',
    'diamond': 'tool',
    'diaper': 'clothing',
    'dice': 'tool',
    'dip': 'food',
    'dirt_bike': 'vehicle',
    'dish': 'tool',
    'dishrag': 'tool',
    'dishwasher': 'tool',
    'dishwashing_liquid': 'tool',
    'diskette': 'tool',
    'diving_board': 'tool',
    'dog': 'animal',
    'dogfood': 'food',
    'doghouse': 'tool',
    'doily': 'tool',
    'doll': 'tool',
    'dollhouse': 'tool',
    'dolly': 'tool',
    'dolphin': 'animal',
    'domino': 'tool',
    'donkey': 'animal',
    'donut': 'food',
    'door': 'tool',
    'doorbell': 'tool',
    'doorhandle': 'tool',
    'doorknob': 'tool',
    'doorknocker': 'tool',
    'doormat': 'tool',
    'doorstop': 'tool',
    'dough': 'food',
    'drain': 'tool',
    'drawer': 'tool',
    'dress': 'clothing',
    'dresser': 'tool',
    'drill': 'tool',
    'drink': 'food',
    'drumstick': 'tool',
    'dryer': 'tool',
    'duck': 'animal',
    'duckling': 'animal',
    'duct': 'tool',
    'duct_tape': 'tool',
    'dumbbell': 'tool',
    'dumbwaiter': 'tool',
    'dumpling': 'food',
    'dumpster': 'tool',
    'duster': 'tool',
    'dustpan': 'tool',
    'dynamite': 'tool',
    'ear': 'tool',
    'earplug': 'tool',
    'earring': 'clothing',
    'earwig': 'animal',
    'easel': 'tool',
    'easter_egg': 'food',
    'eclair': 'food',
    'egg_roll': 'food',
    'eggbeater': 'tool',
    'eggnog': 'food',
    'eggplant': 'food',
    'eggshell': 'food',
    'elbow': 'tool',
    'electric_chair': 'tool',
    'emerald': 'tool',
    'enchilada': 'food',
    'engine': 'tool',
    'envelope': 'tool',
    'eraser': 'tool',
    'exerciser': 'tool',
    'exhaust_pipe': 'tool',
    'extinguisher': 'tool',
    'eye': 'tool',
    'eye_patch': 'clothing',
    'eyedropper': 'tool',
    'eyeliner': 'tool',
    'eyepiece': 'tool',
    'face': 'tool',
    'fan': 'tool',
    'fast_food': 'food',
    'faucet': 'tool',
    'feather': 'tool',
    'fence': 'tool',
    'fencepost': 'tool',
    'fern': 'tool',
    'ferret': 'animal',
    'ferris_wheel': 'tool',
    'fig': 'food',
    'figurine': 'tool',
    'file1': 'tool',
    'file2': 'tool',
    'filing_cabinet': 'tool',
    'film': 'tool',
    'filter': 'tool',
    'finger': 'tool',
    'fingerprint': 'tool',
    'fire': 'tool',
    'fire_alarm': 'tool',
    'fire_hydrant': 'tool',
    'fire_pit': 'tool',
    'firecracker': 'tool',
    'fireplace': 'tool',
    'firetruck': 'vehicle',
    'firewood': 'tool',
    'fireworks': 'tool',
    'first-aid_kit': 'tool',
    'fish': 'animal',
    'fishbowl': 'tool',
    'fishhook': 'tool',
    'fishing_pole': 'tool',
    'fishnet_stockings': 'clothing',
    'flag': 'tool',
    'flagpole': 'tool',
    'flamethrower': 'tool',
    'flan': 'food',
    'flashbulb': 'tool',
    'flashlight': 'tool',
    'flask': 'tool',
    'flatiron': 'tool',
    'flip_flop': 'clothing',
    'flipper': 'tool',
    'float': 'tool',
    'floss': 'tool',
    'flour': 'food',
    'flower': 'tool',
    'flute': 'tool',
    'fly': 'animal',
    'flypaper': 'tool',
    'flyswatter': 'tool',
    'foam': 'tool',
    'fondue': 'food',
    'food_processor': 'tool',
    'foot': 'tool',
    'football': 'tool',
    'football_helmet': 'clothing',
    'footbath': 'tool',
    'footprint': 'tool',
    'footrest': 'tool',
    'forklift': 'vehicle',
    'fossil': 'tool',
    'fountain_pen': 'tool',
    'fox': 'animal',
    'frame': 'tool',
    'french_fries': 'food',
    'frisbee': 'tool',
    'frog': 'animal',
    'fruitcake': 'food',
    'fudge': 'food',
    'fungus': 'tool',
    'funnel': 'tool',
    'fur_coat': 'clothing',
    'furnace': 'tool',
    'fuse': 'tool',
    'gallows': 'tool',
    'game': 'tool',
    'garbage': 'tool',
    'garbage_truck': 'vehicle',
    'gargoyle': 'tool',
    'garter': 'clothing',
    'gas_mask': 'clothing',
    'gasket': 'tool',
    'gate': 'tool',
    'gauge': 'tool',
    'gauze': 'tool',
    'gavel': 'tool',
    'gazelle': 'animal',
    'gear': 'tool',
    'gearshift': 'tool',
    'gel': 'tool',
    'gem': 'tool',
    'generator': 'tool',
    'gift': 'tool',
    'ginger': 'food',
    'gingerbread_man': 'food',
    'giraffe': 'animal',
    'girl': 'tool',
    'glass': 'tool',
    'glasses': 'clothing',
    'globe': 'tool',
    'glue': 'tool',
    'go-kart': 'vehicle',
    'goalpost': 'tool',
    'goat': 'animal',
    'goblet': 'tool',
    'goggles': 'clothing',
    'gold': 'tool',
    'goldfish': 'animal',
    'golf_club': 'tool',
    'gong': 'tool',
    'gourd': 'food',
    'graffiti': 'tool',
    'grain': 'food',
    'gramophone': 'tool',
    'granite': 'tool',
    'granola': 'food',
    'grape': 'food',
    'grapefruit': 'food',
    'grapevine': 'food',
    'grass': 'tool',
    'grate': 'tool',
    'grater': 'tool',
    'gravel': 'tool',
    'gravestone': 'tool',
    'gravy': 'food',
    'green_beans': 'food',
    'grill': 'tool',
    'grille': 'tool',
    'grinder': 'tool',
    'grits': 'food',
    'groundhog': 'animal',
    'guacamole': 'food',
    'guardrail': 'tool',
    'guillotine': 'tool',
    'guinea_pig': 'animal',
    'guitar': 'tool',
    'gum': 'food',
    'gumball': 'food',
    'gumdrop': 'food',
    'gun': 'tool',
    'gurney': 'tool',
    'gutter': 'tool',
    'gyro': 'food',
    'gyroscope': 'tool',
    'hail': 'tool',
    'hair': 'tool',
    'hairbrush': 'tool',
    'hairdryer': 'tool',
    'hairnet': 'clothing',
    'hairpin': 'tool',
    'hairspray': 'tool',
    'ham': 'food',
    'hammock': 'tool',
    'hamster': 'animal',
    'hand': 'tool',
    'handcuff': 'tool',
    'handkerchief': 'clothing',
    'handle': 'tool',
    'handlebar': 'tool',
    'handprint': 'tool',
    'hanger': 'tool',
    'hard_disk': 'tool',
    'harmonica': 'tool',
    'harness': 'tool',
    'harp': 'tool',
    'hash': 'food',
    'hat': 'clothing',
    'hatbox': 'clothing',
    'hatchet': 'tool',
    'hawk': 'animal',
    'hay': 'tool',
    'headband': 'clothing',
    'headdress': 'clothing',
    'headlamp': 'tool',
    'headlight': 'tool',
    'headphones': 'tool',
    'headrest': 'tool',
    'headset': 'tool',
    'hearing_aid': 'tool',
    'hearse': 'vehicle',
    'heater': 'tool',
    'hedge': 'tool',
    'hedgehog': 'animal',
    'helicopter': 'vehicle',
    'helmet': 'clothing',
    'highlighter': 'tool',
    'hinge': 'tool',
    'hip': 'tool',
    'hippopotamus': 'animal',
    'hobbyhorse': 'tool',
    'hockey_stick': 'tool',
    'hoe': 'tool',
    'hole': 'tool',
    'holster': 'clothing',
    'home_plate': 'tool',
    'honey': 'food',
    'honeycomb': 'food',
    'honeypot': 'food',
    'hood': 'clothing',
    'hook1': 'tool',
    'hook2': 'tool',
    'hookah': 'tool',
    'hopscotch': 'tool',
    'horn': 'tool',
    'horse': 'animal',
    'horseshoe': 'tool',
    'hose': 'tool',
    'hot-air_balloon': 'vehicle',
    'hot-water_bottle': 'tool',
    'hot_chocolate': 'food',
    'hot_tub': 'tool',
    'hotdog': 'food',
    'hotplate': 'tool',
    'hourglass': 'tool',
    'hovercraft': 'vehicle',
    'hubcap': 'tool',
    'hula_hoop': 'tool',
    'hummus': 'food',
    'humvee': 'vehicle',
    'hydrant': 'tool',
    'hyena': 'animal',
    'ice': 'food',
    'ice-cream_cone': 'food',
    'ice_cream': 'food',
    'icemaker': 'tool',
    'icepick': 'tool',
    'iceskate': 'clothing',
    'icicle': 'tool',
    'iguana': 'animal',
    'incense': 'tool',
    'incubator': 'tool',
    'inhaler': 'tool',
    'ink': 'tool',
    'inkwell': 'tool',
    'insole': 'clothing',
    'iron': 'tool',
    'ironing_board': 'tool',
    'ivy': 'tool',
    'jack': 'tool',
    'jacket': 'clothing',
    'jackhammer': 'tool',
    'jalapeno': 'food',
    'jam': 'food',
    'jar': 'tool',
    'javelin': 'tool',
    'jeans': 'clothing',
    'jellyfish': 'animal',
    'jersey': 'clothing',
    'jet': 'vehicle',
    'jetski': 'vehicle',
    'jewel': 'tool',
    'jewelry': 'clothing',
    'jigsaw_puzzle': 'tool',
    'joystick': 'tool',
    'jug': 'tool',
    'juice': 'food',
    'juicer1': 'tool',
    'juicer2': 'tool',
    'jump_rope': 'tool',
    'jumpsuit': 'clothing',
    'kale': 'food',
    'kaleidoscope': 'tool',
    'kangaroo': 'animal',
    'kayak': 'vehicle',
    'kazoo': 'tool',
    'kebab': 'food',
    'keg': 'tool',
    'ketchup': 'food',
    'key': 'tool',
    'keyboard': 'tool',
    'keyhole': 'tool',
    'kilt': 'clothing',
    'kimono': 'clothing',
    'kite': 'tool',
    'kitten': 'animal', # 'tool'から修正
    'kiwi': 'food',
    'knee': 'tool',
    'knife': 'tool',
    'knitting': 'tool',
    'knitting_needle': 'tool',
    'knob': 'tool',
    'knot': 'tool',
    'koala': 'animal',
    'lab_coat': 'clothing',
    'ladder': 'tool',
    'ladybug': 'animal',
    'lamb_chop': 'food',
    'lamp': 'tool',
    'lamppost': 'tool',
    'landmine': 'tool',
    'lantern': 'tool',
    'lanyard': 'clothing',
    'laptop': 'tool',
    'lasagna': 'food',
    'laser_pointer': 'tool',
    'latch': 'tool',
    'latte': 'food',
    'lava': 'tool',
    'lavender': 'tool',
    'lawnmower': 'tool',
    'leaf': 'tool',
    'leash': 'tool',
    'lectern': 'tool',
    'leech': 'animal',
    'leek': 'food',
    'leg': 'tool',
    'leggings': 'clothing',
    'lego': 'tool',
    'lemon': 'food',
    'lemonade': 'food',
    'lens': 'tool',
    'leopard': 'animal',
    'leotard': 'clothing',
    'letter_opener': 'tool',
    'license_plate': 'tool',
    'licorice': 'food',
    'lid': 'tool',
    'life_jacket': 'clothing',
    'lifesaver': 'food',
    'light_switch': 'tool',
    'lightbulb': 'tool',
    'lighter': 'tool',
    'lime': 'food',
    'limousine': 'vehicle',
    'lingerie': 'clothing',
    'lion': 'animal',
    'lip_balm': 'tool',
    'lip_gloss': 'tool',
    'lipstick': 'tool',
    'lizard': 'animal',
    'llama': 'animal',
    'lobster': 'animal',
    'lock': 'tool',
    'locker': 'tool',
    'locket': 'clothing',
    'log': 'tool',
    'loincloth': 'clothing',
    'lollipop': 'food',
    'loom': 'tool',
    'loveseat': 'tool',
    'luggage': 'tool',
    'lumber': 'tool',
    'lunchbox': 'tool',
    'macadamia': 'food',
    'macaroni': 'food',
    'machete': 'tool',
    'machine_gun': 'tool',
    'maggot': 'animal',
    'magnet': 'tool',
    'magnifier': 'tool',
    'magnifying_glass': 'tool',
    'mail': 'tool',
    'mailbox': 'tool',
    'makeup': 'tool',
    'mallet': 'tool',
    'man': 'tool',
    'mandolin': 'tool',
    'mango': 'food',
    'manhole': 'tool',
    'mannequin': 'tool',
    'mantle': 'tool',
    'map': 'tool',
    'maple_syrup': 'food',
    'marble': 'tool',
    'margarita': 'food',
    'marker': 'tool',
    'marmalade': 'food',
    'marshmallow': 'food',
    'mascara': 'tool',
    'mashed_potato': 'food',
    'mask': 'clothing',
    'mast': 'tool',
    'mat': 'tool',
    'match': 'tool',
    'matchbox': 'tool',
    'mattress': 'tool',
    'measuring_cup': 'tool',
    'meat': 'food',
    'meat_grinder': 'tool',
    'meatball': 'food',
    'medal': 'clothing',
    'meerkat': 'animal',
    'megaphone': 'tool',
    'melon': 'food',
    'memory_stick': 'tool',
    'metronome': 'tool',
    'microphone': 'tool',
    'microscope': 'tool',
    'microwave': 'tool',
    'milk': 'food',
    'milkshake': 'food',
    'mint': 'food',
    'mirror': 'tool',
    'missile': 'tool',
    'mistletoe': 'tool',
    'mitten': 'clothing',
    'mixer': 'tool',
    'moccasin': 'clothing',
    'mold1': 'tool',
    'mold2': 'tool',
    'mole': 'animal',
    'money': 'tool',
    'mongoose': 'animal',
    'monkey': 'animal',
    'moose': 'animal',
    'mop': 'tool',
    'mosquito_net': 'tool',
    'moss': 'tool',
    'moth': 'animal',
    'motherboard': 'tool',
    'motorcycle': 'vehicle',
    'mouse1': 'animal',
    'mouse2': 'tool',
    'mousepad': 'tool',
    'mousetrap': 'tool',
    'mousse': 'food',
    'mouth': 'tool',
    'mouthpiece': 'tool',
    'mud': 'tool',
    'muffin': 'food',
    'mug': 'tool',
    'mulberry': 'food',
    'mulch': 'tool',
    'mullet': 'animal',
    'mushroom': 'food',
    'mustache': 'tool',
    'mustard': 'food',
    'nacho': 'food',
    'nail': 'tool',
    'nail_clippers': 'tool',
    'nail_file': 'tool',
    'nail_polish': 'tool',
    'napkin': 'tool',
    'napkin_ring': 'tool',
    'navel': 'tool',
    'neck': 'tool',
    'necklace': 'clothing',
    'needle': 'tool',
    'nest': 'tool',
    'net': 'tool',
    'nightshirt': 'clothing',
    'noisemaker': 'tool',
    'noodle': 'food',
    'noose': 'tool',
    'nose': 'tool',
    'notebook': 'tool',
    'notepad': 'tool',
    'nut': 'food',
    'nutcracker': 'tool',
    'oar': 'tool',
    'oatmeal': 'food',
    'octopus': 'animal',
    'odometer': 'tool',
    'oil': 'food',
    'oilcan': 'tool',
    'olive': 'food',
    'orange_rind': 'food',
    'orangutan': 'animal',
    'organ': 'tool',
    'origami': 'tool',
    'otter': 'animal',
    'ottoman': 'tool',
    'outfit': 'clothing',
    'outlet': 'tool',
    'oven': 'tool',
    'overalls': 'clothing',
    'owl': 'animal',
    'oyster': 'animal',
    'pacifier': 'tool',
    'paddle': 'tool',
    'padlock': 'tool',
    'paint': 'tool',
    'paintbrush': 'tool',
    'painting': 'tool',
    'palette': 'tool',
    'pallet': 'tool',
    'palm_tree': 'tool',
    'pan': 'tool',
    'pancake': 'food',
    'panda': 'animal',
    'panties': 'clothing',
    'pants': 'clothing',
    'pantsuit': 'clothing',
    'pantyhose': 'clothing',
    'papaya': 'food',
    'paper': 'tool',
    'paper_bag': 'tool',
    'paper_plate': 'tool',
    'paper_towel': 'tool',
    'paperclip': 'tool',
    'parachute': 'tool',
    'parfait': 'food',
    'parking_meter': 'tool',
    'parrot': 'animal',
    'parsley': 'food',
    'pasta': 'food',
    'pastry': 'food',
    'patch': 'tool',
    'patty': 'food',
    'payphone': 'tool',
    'pea': 'food',
    'peach': 'food',
    'peacock': 'animal',
    'peanut': 'food',
    'peanut_butter': 'food',
    'pearl': 'tool',
    'pecan': 'food',
    'pedal': 'tool',
    'pedometer': 'tool',
    'peeler': 'tool',
    'peg': 'tool',
    'pelican': 'animal',
    'pen': 'tool',
    'pencil': 'tool',
    'pencil_sharpener': 'tool',
    'pendulum': 'tool',
    'penguin': 'animal',
    'penholder': 'tool',
    'penlight': 'tool',
    'pennant': 'tool',
    'pepper2': 'food',
    'pepper_mill': 'tool',
    'peppermint': 'food',
    'pepperoni': 'food',
    'perfume': 'tool',
    'periscope': 'tool',
    'pesto': 'food',
    'pet_food': 'food',
    'petal': 'tool',
    'petri_dish': 'tool',
    'phone': 'tool',
    'phone_booth': 'tool',
    'photo_booth': 'tool',
    'photograph': 'tool',
    'piano': 'tool',
    'pickle': 'food',
    'piecrust': 'food',
    'pig': 'animal',
    'piggy_bank': 'tool',
    'pill': 'tool',
    'pillbox': 'tool',
    'pillow': 'tool',
    'pin': 'tool',
    'pinball': 'tool',
    'pincushion': 'tool',
    'pine_needle': 'tool',
    'pine_tree': 'tool',
    'pineapple': 'food',
    'pinecone': 'tool',
    'ping-pong_table': 'tool',
    'pinwheel': 'tool',
    'pipe1': 'tool',
    'pipe2': 'tool',
    'pistachio': 'food',
    'pita': 'food',
    'pitcher': 'tool',
    'pitchfork': 'tool',
    'pizza': 'food',
    'place_mat': 'tool',
    'plant': 'tool',
    'plaster_cast': 'tool',
    'plastic_film': 'tool',
    'plate': 'tool',
    'platypus': 'animal',
    'playing_card': 'tool',
    'playpen': 'tool',
    'pliers': 'tool',
    'plug': 'tool',
    'plum': 'food',
    'plunger': 'tool',
    'pocket_watch': 'clothing',
    'pogo_stick': 'tool',
    'poinsettia': 'tool',
    'poker': 'tool',
    'polar_bear': 'animal',
    'polaroid': 'tool',
    'pole': 'tool',
    'police_car': 'vehicle',
    'polisher': 'tool',
    'polo_shirt': 'clothing',
    'polygraph': 'tool',
    'pom-pom': 'tool',
    'pomegranate': 'food',
    'pony': 'animal',
    'poodle': 'animal',
    'pool_table': 'tool',
    'poppy': 'tool',
    'porcupine': 'animal',
    'porthole': 'tool',
    'poster': 'tool',
    'pot': 'tool',
    'potato': 'food',
    'potholder': 'tool',
    'pothole': 'tool',
    'potpie': 'food',
    'potpourri': 'tool',
    'powder': 'tool',
    'power_line': 'tool',
    'praying_mantis': 'animal',
    'printer': 'tool',
    'prism': 'tool',
    'projector': 'tool',
    'propeller': 'tool',
    'prune': 'food',
    'puck': 'tool',
    'pudding': 'food',
    'puddle': 'tool',
    'puffin': 'animal',
    'pulley': 'tool',
    'pulpit': 'tool',
    'pump': 'tool',
    'pumpkin': 'food',
    'punch1': 'tool',
    'punching_bag': 'tool',
    'puppet': 'tool',
    'puppy': 'animal',
    'quad': 'vehicle',
    'quesadilla': 'food',
    'quiche': 'food',
    'quill': 'tool',
    'quilt': 'tool',
    'rabbit': 'animal',
    'raccoon': 'animal',
    'racehorse': 'animal',
    'rack1': 'tool',
    'rack2': 'tool',
    'racket': 'tool',
    'radar': 'tool',
    'radiator': 'tool',
    'radio': 'tool',
    'raft': 'vehicle',
    'rag': 'tool',
    'railing': 'tool',
    'rain_gauge': 'tool',
    'raincoat': 'clothing',
    'raisin': 'food',
    'rake': 'tool',
    'ram': 'animal',
    'ramp': 'tool',
    'rat': 'animal',
    'ratchet': 'tool',
    'rattle': 'tool',
    'rattlesnake': 'animal',
    'ravioli': 'food',
    'razor': 'tool',
    'razor_blade': 'tool',
    'ready_meal': 'food',
    'rearview_mirror': 'tool',
    'recliner': 'tool',
    'record': 'tool',
    'record_player': 'tool',
    'red_carpet': 'tool',
    'reel': 'tool',
    'refrigerator': 'tool',
    'reindeer': 'animal',
    'remote_control': 'tool',
    'retainer': 'tool',
    'revolver': 'tool',
    'revolving_door': 'tool',
    'rhubarb': 'food',
    'ribbon': 'tool',
    'rice': 'food',
    'rickshaw': 'vehicle',
    'rifle': 'tool',
    'rim': 'tool',
    'ring': 'clothing',
    'riser': 'tool',
    'road_sign': 'tool',
    'roadsweeper': 'vehicle',
    'robe': 'clothing',
    'rock': 'tool',
    'rocket': 'vehicle',
    'rocking_chair': 'tool',
    'rocking_horse': 'tool',
    'roll': 'food',
    'roller': 'tool',
    'roller_coaster': 'vehicle',
    'rollerblade': 'clothing',
    'rollerskate': 'clothing',
    'rolling_pin': 'tool',
    'roof_rack': 'tool',
    'root': 'tool',
    'rope': 'tool',
    'rosary': 'clothing',
    'rose': 'tool',
    'roulette_wheel': 'tool',
    'router': 'tool',
    'rubber_band': 'tool',
    'ruby': 'tool',
    'rudder': 'tool',
    'ruler': 'tool',
    'rust': 'tool',
    'saddle': 'tool',
    'safety_pin': 'tool',
    'saffron': 'food',
    'sail': 'tool',
    'salad': 'food',
    'salami': 'food',
    'saltshaker': 'tool',
    'sand': 'tool',
    'sandbag': 'tool',
    'sandbox': 'tool',
    'sandcastle': 'tool',
    'sandwich': 'food',
    'sarcophagus': 'tool',
    'sardine': 'animal',
    'satellite': 'tool',
    'satellite_dish': 'tool',
    'sauce': 'food',
    'saucer': 'tool',
    'sauerkraut': 'food',
    'saw': 'tool',
    'sawhorse': 'tool',
    'saxophone': 'tool',
    'scaffold': 'tool',
    'scaffolding': 'tool',
    'scale': 'tool',
    'scalpel': 'tool',
    'scanner': 'tool',
    'scarecrow': 'tool',
    'scarf': 'clothing',
    'school_bus': 'vehicle',
    'scissors': 'tool',
    'scone': 'food',
    'scoop': 'tool',
    'scoreboard': 'tool',
    'scorpion': 'animal',
    'scrabble': 'tool',
    'scrambled_egg': 'food',
    'scraper': 'tool',
    'screen1': 'tool',
    'screen2': 'tool',
    'screw': 'tool',
    'screwdriver': 'tool',
    'scuba': 'clothing',
    'sea_urchin': 'animal',
    'seafood': 'food',
    'seahorse': 'animal',
    'seal': 'animal',
    'seaplane': 'vehicle',
    'seatbelt': 'tool',
    'seesaw': 'tool',
    'seismograph': 'tool',
    'sequin': 'clothing',
    'sewage': 'tool',
    'sewing_kit': 'tool',
    'sewing_machine': 'tool',
    'shaker': 'tool',
    'shark': 'animal',
    'shaving_cream': 'tool',
    'shawl': 'clothing',
    'shears': 'tool',
    'sheath': 'tool',
    'sheep': 'animal',
    'sheet': 'tool',
    'shelf': 'tool',
    'shell1': 'tool',
    'shell2': 'tool',
    'shell3': 'tool',
    'shield': 'tool',
    'ship': 'vehicle',
    'shirt': 'clothing',
    'shoe': 'clothing',
    'shoe_polish': 'tool',
    'shoehorn': 'tool',
    'shoelace': 'clothing',
    'shopping_basket': 'tool',
    'shopping_cart': 'tool',
    'shortbread': 'food',
    'shorts': 'clothing',
    'shoulder': 'tool',
    'shovel': 'tool',
    'shower': 'tool',
    'shower_cap': 'clothing',
    'shower_curtain': 'tool',
    'showerhead': 'tool',
    'shredder': 'tool',
    'shrimp': 'animal',
    'shuffleboard': 'tool',
    'shutter': 'tool',
    'sickle': 'tool',
    'sidecar': 'vehicle',
    'sifter': 'tool',
    'silicone': 'tool',
    'silverware': 'tool',
    'sim_card': 'tool',
    'sink': 'tool',
    'siren': 'tool',
    'skeleton': 'tool',
    'skewer': 'tool',
    'ski': 'tool',
    'ski_boots': 'clothing',
    'ski_lift': 'tool',
    'ski_pole': 'tool',
    'skin': 'tool',
    'skirt': 'clothing',
    'skull': 'tool',
    'skunk': 'animal',
    'sledgehammer': 'tool',
    'slicer': 'tool',
    'slime': 'tool',
    'sling': 'tool',
    'slipper': 'clothing',
    'slot': 'tool',
    'slot_machine': 'tool',
    'sloth': 'animal',
    'slug': 'animal',
    'smoke_alarm': 'tool',
    'smoothie': 'food',
    'snack': 'food',
    'snail': 'animal',
    'snake': 'animal',
    'snorkel': 'tool',
    'snow': 'tool',
    'snowball': 'tool',
    'snowboard': 'tool',
    'snowman': 'tool',
    'snowmobile': 'vehicle',
    'snowplow': 'vehicle',
    'snowsuit': 'clothing',
    'soap': 'tool',
    'soccer_ball': 'tool',
    'sock': 'clothing',
    'soda': 'food',
    'soda_fountain': 'tool',
    'sofa_bed': 'tool',
    'solar_panel': 'tool',
    'soldering_iron': 'tool',
    'sombrero': 'clothing',
    'sonogram': 'tool',
    'sorbet': 'food',
    'souffle': 'food',
    'soup': 'food',
    'soy_sauce': 'food',
    'space_shuttle': 'vehicle',
    'spacesuit': 'clothing',
    'spaghetti': 'food',
    'spam': 'food',
    'spareribs': 'food',
    'spark_plug': 'tool',
    'sparkler': 'tool',
    'speaker': 'tool',
    'spear': 'tool',
    'speedboat': 'vehicle',
    'speedometer': 'tool',
    'spider': 'animal',
    'spider_web': 'tool',
    'spinach': 'food',
    'splinter': 'tool',
    'sponge': 'tool',
    'spool': 'tool',
    'spout': 'tool',
    'spring_roll': 'food',
    'springboard': 'tool',
    'sprinkler': 'tool',
    'sprouts': 'food',
    'spur': 'tool',
    'squash': 'food',
    'squeegee': 'tool',
    'squid': 'animal',
    'squirrel': 'animal',
    'squirt_gun': 'tool',
    'stained_glass': 'tool',
    'stair': 'tool',
    'stake': 'tool',
    'stalagmite': 'tool',
    'stamp1': 'tool',
    'stamp2': 'tool',
    'staple': 'tool',
    'staple_gun': 'tool',
    'stapler': 'tool',
    'star_fruit': 'food',
    'starfish': 'animal',
    'statue': 'tool',
    'steak': 'food',
    'steamroller': 'vehicle',
    'steering_wheel': 'tool',
    'stem': 'tool',
    'step_stool': 'tool',
    'stereo': 'tool',
    'stew': 'food',
    'stick': 'tool',
    'sticker': 'tool',
    'stiletto': 'clothing',
    'stilt': 'tool',
    'stingray': 'animal',
    'stir_fry': 'food',
    'stirrup': 'tool',
    'stockings': 'clothing',
    'stomach': 'tool',
    'stool': 'tool',
    'stopwatch': 'tool',
    'stove1': 'tool',
    'stove2': 'tool',
    'straightjacket': 'clothing',
    'strainer': 'tool',
    'strap': 'tool',
    'straw1': 'tool',
    'straw2': 'tool',
    'streetlight': 'tool',
    'stretcher': 'tool',
    'string_cheese': 'food',
    'stroller': 'tool',
    'stuffing': 'food',
    'stump': 'tool',
    'subway': 'vehicle',
    'sugar_cube': 'food',
    'suitcase': 'tool',
    'sundae': 'food',
    'sundial': 'tool',
    'sunflower': 'tool',
    'sunglasses': 'clothing',
    'sunroof': 'tool',
    'surfboard': 'tool',
    'sushi': 'food',
    'suspenders': 'clothing',
    'swab': 'tool',
    'swan': 'animal',
    'sweater': 'clothing',
    'sweatsuit': 'clothing',
    'sweeper': 'tool',
    'sweet_potato': 'food',
    'swimming_pool': 'tool',
    'swimsuit': 'clothing',
    'swing': 'tool',
    'swing_set': 'tool',
    'switch': 'tool',
    'swizzle_stick': 'tool',
    'sword': 'tool',
    'swordfish': 'animal',
    'syringe': 'tool',
    'syrup': 'food',
    'tab': 'tool',
    'tablecloth': 'tool',
    'tablet': 'tool',
    'tack': 'tool',
    'tackle': 'tool',
    'taco': 'food',
    'tadpole': 'animal',
    'taffy': 'food',
    'tag': 'tool',
    'tamale': 'food',
    'tambourine': 'tool',
    'tank1': 'tool',
    'tank2': 'vehicle',
    'tape': 'tool',
    'tape_measure': 'tool',
    'tapestry': 'tool',
    'tarantula': 'animal',
    'target': 'tool',
    'tarp': 'tool',
    'tattoo': 'tool',
    'taxi': 'vehicle',
    'tea': 'food',
    'teabag': 'food',
    'teacup': 'tool',
    'teapot': 'tool',
    'teddy_bear': 'tool',
    'tee': 'clothing',
    'teepee': 'tool',
    'telegraph': 'tool',
    'telephone_pole': 'tool',
    'telescope': 'tool',
    'tennis_ball': 'tool',
    'tent': 'tool',
    'terrarium': 'tool',
    'test_tube': 'tool',
    'thermometer': 'tool',
    'thermos': 'tool',
    'thermostat': 'tool',
    'thimble': 'tool',
    'thorn': 'tool',
    'thread': 'tool',
    'throne': 'tool',
    'thumb': 'tool',
    'thumbtack': 'tool',
    'ticktacktoe': 'tool',
    'tie': 'clothing',
    'tiger': 'animal',
    'tile': 'tool',
    'timer': 'tool',
    'tinsel': 'tool',
    'tiramisu': 'food',
    'toad': 'animal',
    'toast': 'food',
    'toaster': 'tool',
    'toaster_oven': 'tool',
    'toe': 'tool',
    'toga': 'clothing',
    'toilet': 'tool',
    'toilet_paper': 'tool',
    'tomato': 'food',
    'tongue': 'tool',
    'toolbox': 'tool',
    'tooth': 'tool',
    'toothbrush': 'tool',
    'toothpaste': 'tool',
    'toothpick': 'tool',
    'torch': 'tool',
    'torpedo': 'tool',
    'torso': 'tool',
    'tortellini': 'food',
    'tortilla': 'food',
    'tostada': 'food',
    'totem_pole': 'tool',
    'toucan': 'animal',
    'touchpad': 'tool',
    'towel': 'tool',
    'towel_rack': 'tool',
    'toy': 'tool',
    'tractor': 'vehicle',
    'traffic_light': 'tool',
    'trailer': 'vehicle',
    'train': 'vehicle',
    'train_car': 'vehicle',
    'train_set': 'vehicle',
    'trampoline': 'tool',
    'trap': 'tool',
    'trapdoor': 'tool',
    'trashcan': 'tool',
    'tray': 'tool',
    'treasure': 'tool',
    'tree': 'tool',
    'tree_trunk': 'tool',
    'triangle': 'tool',
    'tricycle': 'vehicle',
    'trident': 'tool',
    'trigger': 'tool',
    'tripod': 'tool',
    'trolley': 'vehicle',
    'trombone': 'tool',
    'trophy': 'tool',
    'trough': 'tool',
    'trowel': 'tool',
    'truck': 'vehicle',
    'trumpet': 'tool',
    'trunk': 'tool',
    'tuba': 'tool',
    'tugboat': 'vehicle',
    'tulip': 'tool',
    'tumbleweed': 'tool',
    'tuning_fork': 'tool',
    'tupperware': 'tool',
    'turban': 'clothing',
    'turbine': 'tool',
    'turf': 'tool',
    'turnstile': 'tool',
    'turntable': 'tool',
    'turtle': 'animal',
    'turtleneck': 'clothing',
    'tuxedo': 'clothing',
    'tweezers': 'tool',
    'twig': 'tool',
    'typewriter': 'tool',
    'ukulele': 'tool',
    'umbrella': 'tool',
    'undershirt': 'clothing',
    'underwear': 'clothing',
    'uniform': 'clothing',
    'urinal': 'tool',
    'urn': 'tool',
    'vacuum': 'tool',
    'valve': 'tool',
    'van': 'vehicle',
    'vase': 'tool',
    'vegetable': 'food',
    'veil': 'clothing',
    'velcro': 'tool',
    'vending_machine': 'tool',
    'vent': 'tool',
    'vest': 'clothing',
    'vial': 'tool',
    'videocassette': 'tool',
    'videogame': 'tool',
    'viewfinder': 'tool',
    'violin': 'tool',
    'visor': 'clothing',
    'vulture': 'animal',
    'wafer': 'food',
    'waffle': 'food',
    'waffle_iron': 'tool',
    'wagon': 'vehicle',
    'walker1': 'tool',
    'walker2': 'tool',
    'wall': 'tool',
    'wallet': 'clothing',
    'walrus': 'animal',
    'wand': 'tool',
    'warthog': 'animal',
    'washboard': 'tool',
    'washcloth': 'tool',
    'washing_machine': 'tool',
    'wasp': 'animal',
    'watch': 'clothing',
    'water_bottle': 'tool',
    'water_cooler': 'tool',
    'water_filter': 'tool',
    'water_fountain': 'tool',
    'water_heater': 'tool',
    'watering_can': 'tool',
    'watermelon': 'food',
    'waterwheel': 'tool',
    'wax': 'tool',
    'wax_paper': 'tool',
    'weasel': 'animal',
    'weather_vane': 'tool',
    'webcam': 'tool',
    'wedding_cake': 'food',
    'wedge': 'tool',
    'weed': 'tool',
    'wetsuit': 'clothing',
    'whale': 'animal',
    'wheel': 'tool',
    'wheelbarrow': 'vehicle',
    'whip': 'tool',
    'whipped_cream': 'food',
    'whisk': 'tool',
    'whistle': 'tool',
    'whiteboard': 'tool',
    'whoopee_cushion': 'tool',
    'wick': 'tool',
    'wig': 'clothing',
    'wind_chimes': 'tool',
    'window': 'tool',
    'windowsill': 'tool',
    'windshield_wiper': 'tool',
    'windsock': 'tool',
    'wine_bottle': 'tool',
    'wine_cooler': 'tool',
    'wineglass': 'tool',
    'wing': 'tool',
    'wire': 'tool',
    'wire_cutters': 'tool',
    'wolf': 'animal',
    'woman': 'tool',
    'wood': 'tool',
    'wooden_leg': 'tool',
    'workbench': 'tool',
    'worm': 'animal',
    'wrap': 'tool',
    'wrapping_paper': 'tool',
    'wreath': 'tool',
    'wreck': 'tool',
    'wrench': 'tool',
    'wrist': 'tool',
    'xylophone': 'tool',
    'yacht': 'vehicle',
    'yak': 'animal',
    'yarn': 'tool',
    'yo-yo': 'tool',
    'yogurt': 'food',
    'yoke': 'tool',
    'yolk': 'food',
    'zebra': 'animal',
    'zipper': 'clothing',
    'zucchini': 'food'
}


In [ ]:
# omnicampus 実行用
!pip install ipywidgets
!pip install transformers
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 23.1.2 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 23.1.2 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://download.pytorch.org/whl/cu121, https://pypi.ngc.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 92.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 96.7 MB/s eta 0:00:00:00:01
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 66.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.16.0a0
    Uninstalling t

In [ ]:
# ライブラリのインポートとシード固定
import os, sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import zipfile
import shutil
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.transforms as transforms
from einops.layers.torch import Rearrange
from einops import repeat, rearrange
from glob import glob
from termcolor import cprint
from tqdm.notebook import tqdm
from PIL import Image


SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
# ドライブのマウント（Colabの場合）
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ワーキングディレクトリを作成し移動．ノートブックを配置したディレクトリに適宜書き換え
WORK_DIR = "/workspace/assets"
os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

/workspace/assets


In [ ]:
zip_path = "training_images.zip"
extract_dir = "extracted_training_images"

In [ ]:
# import zipfile

# zip_path = "/workspace/assets/extracted_training_images/training_images/00294_chipmunk.zip"  # ZIPファイルのパス
# extract_to = "."  # 今のディレクトリに展開

# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(path=extract_to)


In [ ]:
target_dir = '/workspace/assets/extracted_training_images/training_images'

subfolders = [name for name in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, name))]

print(f"Folder number: {len(subfolders)}")

Folder number: 1655


## 2.データセット

ノートブックと同じディレクトリに`data/`が存在することを確認してください．

In [ ]:
class EEGAugmentation:
    """EEG data augmentation"""

    def __init__(self,
                 noise_std=0.02,
                 time_shift_range=0.1,
                 amplitude_scale_range=(0.9, 1.1),
                 channel_dropout_prob=0.1,
                 time_mask_ratio=0.05):
        self.noise_std = noise_std
        self.time_shift_range = time_shift_range
        self.amplitude_scale_range = amplitude_scale_range
        self.channel_dropout_prob = channel_dropout_prob
        self.time_mask_ratio = time_mask_ratio

    def add_gaussian_noise(self, x):
        noise = torch.randn_like(x) * self.noise_std
        return x + noise

    def time_shift(self, x):
        batch_size, channels, seq_len = x.shape
        shift_range = int(seq_len * self.time_shift_range)

        if shift_range > 0:
            shift = random.randint(-shift_range, shift_range)
            if shift > 0:
                x = torch.cat([x[:, :, shift:], x[:, :, :shift]], dim=2)
            elif shift < 0:
                x = torch.cat([x[:, :, shift:], x[:, :, :shift]], dim=2)
        return x

    def amplitude_scaling(self, x):
        scale = random.uniform(*self.amplitude_scale_range)
        return x * scale

    def channel_dropout(self, x):
        batch_size, channels, seq_len = x.shape
        mask = torch.rand(batch_size, channels, 1) > self.channel_dropout_prob
        return x * mask.to(x.device)

    def time_masking(self, x):
        batch_size, channels, seq_len = x.shape
        mask_len = int(seq_len * self.time_mask_ratio)

        if mask_len > 0:
            start_idx = random.randint(0, seq_len - mask_len)
            x[:, :, start_idx:start_idx + mask_len] = 0
        return x

    def frequency_masking(self, x):
        batch_size, channels, seq_len = x.shape
        mask_channels = int(channels * 0.1)  # 10%のチャンネルをマスク

        if mask_channels > 0:
            start_ch = random.randint(0, channels - mask_channels)
            x[:, start_ch:start_ch + mask_channels, :] = 0
        return x

    def mixup_augmentation(self, x1, x2, y1, y2, alpha=0.2):
        lambda_mix = np.random.beta(alpha, alpha)
        mixed_x = lambda_mix * x1 + (1 - lambda_mix) * x2
        return mixed_x, y1, y2, lambda_mix

    def __call__(self, x, augment_prob=0.8):
        if random.random() < augment_prob:
            # use data augmentation randomly
            augmentations = [
                self.add_gaussian_noise,
                self.time_shift,
                self.amplitude_scaling,
                self.channel_dropout,
                self.time_masking,
                self.frequency_masking
            ]

            # choose 2-3 augmentation
            num_augs = random.randint(2, 3)
            selected_augs = random.sample(augmentations, num_augs)

            for aug in selected_augs:
                x = aug(x)

        return x

In [ ]:
class ThingsImageDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        self.category_mapping = {
            'animal': 0, 'food': 1, 'clothing': 2, 'tool': 3, 'vehicle': 4
        }

        self.detailed_mapping = DETAILED_MAPPING

        self._load_images()

    def _load_images(self):
        for folder_name in sorted(os.listdir(self.image_dir)):
            folder_path = os.path.join(self.image_dir, folder_name)
#             print(f"Folder path: {folder_path}")
            if os.path.isdir(folder_path):
                # 00001_aardvark -> aardvark
                class_name = folder_name.split('_', 1)[-1].lower()

                category = self._get_category(class_name)
                if category is not None:
                    label = self.category_mapping[category]

                    for img_name in os.listdir(folder_path):
                        if img_name.lower().endswith(('.jpg')):
                            img_path = os.path.join(folder_path, img_name)
                            self.image_paths.append(img_path)
                            self.labels.append(label)

    def _get_category(self, class_name):
        if class_name in self.detailed_mapping:
            return self.detailed_mapping[class_name]
        else:
            print(f"Error: Class name {class_name} is not in detailed map. ")
            return None
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
class ThingsEEGDataset(torch.utils.data.Dataset):
    def __init__(self, split: str, apply_whitening: bool = False, normalize_per_subject: bool = True) -> None:
        super().__init__()
        assert split in ["train", "val", "test"], f"Invalid split: {split}"
        self.split = split
        self.num_classes = 5
        self.num_subjects = 10
        self.normalize_per_subject = normalize_per_subject

        self.X = np.load(f"EEG/{split}/eeg.npy")
        self.X = torch.from_numpy(self.X).to(torch.float32)
        self.subject_idxs = np.load(f"EEG/{split}/subject_idxs.npy")
        self.subject_idxs = torch.from_numpy(self.subject_idxs)

        if split in ["train", "val"]:
            self.y = np.load(f"EEG/{split}/labels.npy")
            self.y = torch.from_numpy(self.y)

        # 被験者ごとの正規化
        if self.normalize_per_subject:
            self._normalize_per_subject()
        else:
            # 全体での正規化
            self.X = (self.X - self.X.mean(dim=-1, keepdim=True)) / (self.X.std(dim=-1, keepdim=True) + 1e-5)

        self.apply_whitening = apply_whitening
        if self.apply_whitening:
            self._apply_whitening()

        self.num_subjects = len(torch.unique(self.subject_idxs))
        print(f"EEG: {self.X.shape}, labels: {self.y.shape if hasattr(self, 'y') else None}, subject indices: {self.subject_idxs.shape}")
        print(f"Number of subjects: {self.num_subjects}")

    def _normalize_per_subject(self):
        """被験者ごとの正規化"""
        for subject_id in torch.unique(self.subject_idxs):
            mask = self.subject_idxs == subject_id
            subject_data = self.X[mask]
            # チャネルごとに正規化
            mean = subject_data.mean(dim=(0, 2), keepdim=True)
            std = subject_data.std(dim=(0, 2), keepdim=True)
            self.X[mask] = (subject_data - mean) / (std + 1e-5)

    def _apply_whitening(self):
        """ホワイトニング処理"""
        X_flat = self.X.reshape(self.X.shape[0], -1).numpy()
        X_mean = X_flat.mean(axis=0, keepdims=True)
        X_centered = X_flat - X_mean
        cov = np.cov(X_centered, rowvar=False)

        U, S, Vt = np.linalg.svd(cov)

        S_clipped = np.maximum(S, 1e-3)
        self.whitening_matrix = torch.from_numpy(U @ np.diag(1.0 / np.sqrt(S_clipped)) @ U.T).to(torch.float32)
        self.X_mean = torch.from_numpy(X_mean).to(torch.float32)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, i):
        X = self.X[i]
        if self.apply_whitening:
            X_flat = X.flatten()
            X_flat = X_flat - self.X_mean.squeeze()
            X_flat = torch.matmul(self.whitening_matrix, X_flat)
            X = X_flat.view_as(X)

        if hasattr(self, "y"):
            return X, self.y[i], self.subject_idxs[i]
        else:
            return X, self.subject_idxs[i]

    @property
    def num_channels(self) -> int:
        return self.X.shape[1]

    @property
    def seq_len(self) -> int:
        return self.X.shape[2]

In [ ]:
class AugmentedThingsEEGDataset(Dataset):
    """Dataset with augmentation"""

    def __init__(self, original_dataset, augmentation=None, use_mixup=False):
        self.original_dataset = original_dataset
        self.augmentation = augmentation
        self.use_mixup = use_mixup

    def __len__(self):
        return len(self.original_dataset)

    def __getitem__(self, idx):
        x, y, subject_idx = self.original_dataset[idx]

        # basic augmentation
        if self.augmentation is not None:
            x = self.augmentation(x.unsqueeze(0)).squeeze(0)

        # Mixup
        if self.use_mixup and random.random() < 0.3:
            # choose another sample
            mix_idx = random.randint(0, len(self.original_dataset) - 1)
            x2, y2, _ = self.original_dataset[mix_idx]

            if self.augmentation is not None:
                x2 = self.augmentation(x2.unsqueeze(0)).squeeze(0)

            # Mixup
            mixed_x, y1, y2, lambda_mix = self.augmentation.mixup_augmentation(
                x.unsqueeze(0), x2.unsqueeze(0), y, y2
            )

            return mixed_x.squeeze(0), (y1, y2, lambda_mix), subject_idx

        return x, y, subject_idx

## 3.ベースラインモデル

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=256, pretrained=True):
        super().__init__()
        self.backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, embed_dim)
        self.embed_dim = embed_dim

    def forward(self, x):
        return F.normalize(self.backbone(x), dim=-1)

In [ ]:
# Conformer-inspired architecture for EEG data
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        output = torch.matmul(attention_weights, V)
        return output

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        attention = self.scaled_dot_product_attention(Q, K, V, mask)
        attention = attention.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        output = self.W_o(attention)
        return output

class ConvolutionModule(nn.Module):
    def __init__(self, d_model, kernel_size=31, dropout=0.1):
        super().__init__()
        self.pointwise_conv1 = nn.Conv1d(d_model, d_model * 2, kernel_size=1)
        self.depthwise_conv = nn.Conv1d(d_model, d_model, kernel_size=kernel_size,
                                       padding=(kernel_size - 1) // 2, groups=d_model)
        self.batch_norm = nn.BatchNorm1d(d_model)
        self.pointwise_conv2 = nn.Conv1d(d_model, d_model, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x.transpose(1, 2)  # (batch, d_model, seq_len)

        x = self.pointwise_conv1(x)
        x = F.glu(x, dim=1)
        x = self.depthwise_conv(x)
        x = self.batch_norm(x)
        x = F.silu(x)
        x = self.pointwise_conv2(x)
        x = self.dropout(x)

        return x.transpose(1, 2)  # (batch, seq_len, d_model)

class ConformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, conv_kernel_size=31, dropout=0.1):
        super().__init__()
        self.ff1 = FeedForward(d_model, d_ff, dropout)
        self.mha = MultiHeadAttention(d_model, num_heads, dropout)
        self.conv = ConvolutionModule(d_model, conv_kernel_size, dropout)
        self.ff2 = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.norm4 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Feed forward 1
        x = x + 0.5 * self.dropout(self.ff1(self.norm1(x)))

        # Multi-head attention
        x = x + self.dropout(self.mha(self.norm2(x), self.norm2(x), self.norm2(x)))

        # Convolution
        x = x + self.dropout(self.conv(self.norm3(x)))

        # Feed forward 2
        x = x + 0.5 * self.dropout(self.ff2(self.norm4(x)))

        return x

class EEGEncoder(nn.Module):
    def __init__(self, seq_len, in_channels, embed_dim=256, d_model=256, num_heads=8,
                 num_layers=4, d_ff=1024, dropout=0.2):
        super().__init__()

        self.embed_dim = embed_dim

        self.augmentation = EEGAugmentation()

        # Input projection
        self.input_projection = nn.Conv1d(in_channels, d_model, kernel_size=3, padding=1)

        # Positional encoding
        self.pos_encoding = nn.Parameter(torch.randn(1, seq_len, d_model))

        # Conformer blocks
        self.conformer_blocks = nn.ModuleList([
            ConformerBlock(d_model, num_heads, d_ff, dropout=dropout)
            for _ in range(num_layers)
        ])

        # Classification head
        self.norm = nn.LayerNorm(d_model)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.projection = nn.Linear(d_model, embed_dim)

    def forward(self, x):
        # x: (batch, channels, seq_len)
        batch_size = x.size(0)

        # Input projection
        x = self.input_projection(x)  # (batch, d_model, seq_len)
        x = x.transpose(1, 2)  # (batch, seq_len, d_model)

        # Add positional encoding
        x = x + self.pos_encoding

        # Apply conformer blocks
        for block in self.conformer_blocks:
            x = block(x)

        # Normalization
        x = self.norm(x)

        x = x.transpose(1, 2)  # (batch, d_model, seq_len)
        x = self.global_pool(x).squeeze(-1)  # (batch, d_model)
        x = self.projection(x)
        x = F.normalize(x, dim=-1)

        return x

class EEGClassifier(nn.Module):
    def __init__(self, clip_model, num_classes, freeze_encoder=False):
        super().__init__()
        self.eeg_encoder = clip_model.eeg_encoder
        self.classifier = nn.Sequential(
            nn.Linear(clip_model.eeg_encoder.embed_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

        if freeze_encoder:
            for param in self.eeg_encoder.parameters():
                param.requires_grad = False

    def forward(self, x):
        features = self.eeg_encoder(x)

        return self.classifier(features)

In [ ]:
class EEGImageCLIP(nn.Module):
    def __init__(self, seq_len, in_channels, embed_dim=256, temperature=0.07):
        super().__init__()
        self.eeg_encoder = EEGEncoder(seq_len, in_channels, embed_dim)
        self.image_encoder = ImageEncoder(embed_dim)
        self.temperature = temperature

    def forward(self, eeg, images=None):
        eeg_features = self.eeg_encoder(eeg)

        if images is not None:
            image_features = self.image_encoder(images)
            return eeg_features, image_features

        return eeg_features

## 4.訓練実行

In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, eeg_features, image_features):
        similarity_matrix = torch.matmul(eeg_features, image_features.T) / self.temperature

        batch_size = eeg_features.size(0)
        labels = torch.arange(batch_size).to(eeg_features.device)

        loss_eeg_to_image = F.cross_entropy(similarity_matrix, labels)
        loss_image_to_eeg = F.cross_entropy(similarity_matrix.T, labels)

        return (loss_eeg_to_image + loss_image_to_eeg) / 2

In [ ]:
def mixup_criterion(criterion, pred, y_mixed):
    """criterion for mixup"""
    y1, y2, lambda_mix = y_mixed
    return lambda_mix * criterion(pred, y1) + (1 - lambda_mix) * criterion(pred, y2)

In [ ]:
from torch.cuda.amp import autocast, GradScaler

def pretrain_clip_model(model, eeg_loader, image_loader, epochs=20, lr=1e-4, clip_start_epoch=0):
    """Pre-train the model using reconstruction and contrastive learning"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    contrastive_loss = ContrastiveLoss()

    scaler = GradScaler()

    best_loss = float('inf')

    if clip_start_epoch >= 1:
        checkpoint = torch.load("clip_model_last.pt")
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        best_loss = checkpoint['best_loss']

    model.train()

    for epoch in range(clip_start_epoch, epochs):
        total_loss = 0
        num_batches = 0

        eeg_iter = iter(eeg_loader)
        image_iter = iter(image_loader)

        num_batches_estimate = min(len(eeg_loader), len(image_loader))
        pbar = tqdm(total=num_batches_estimate, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

        while True:
            try:
                eeg_batch = next(eeg_iter)
                image_batch = next(image_iter)
            except StopIteration:
                break

            eeg_data, eeg_labels, _ = eeg_batch
            image_data, image_labels = image_batch

            eeg_labels_np = np.array(eeg_labels)
            img_labels_np = np.array(image_labels)

            matching = np.where(eeg_labels_np[:, None] == img_labels_np[None, :])
            if len(matching[0]) == 0:
                continue

            eeg_indices, img_indices = matching
            matched_eeg = eeg_data[eeg_indices].to("cuda")
            matched_images = image_data[img_indices].to("cuda")

            optimizer.zero_grad()

            with autocast():
                eeg_features, image_features = model(matched_eeg, matched_images)

                loss = contrastive_loss(eeg_features, image_features)


            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            num_batches += 1
            pbar.update(1)

        pbar.close()
        scheduler.step()

        avg_loss = total_loss / max(num_batches, 1)
        print(f"CLIP Epoch {epoch+1} / {epochs} | Loss: {avg_loss:.4f}")

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_loss': best_loss,
        }, "clip_model_last.pt")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), "clip_pretrained_model.pt")

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    return model


In [ ]:
def train_with_pretraining(clip_start_epoch=0, finetune_start_epoch=0):

    batch_size = 32
    clip_epochs = 30
    finetune_epochs = 80
    clip_lr = 1e-4
    finetune_lr =1e-5

    # augmentation
    eeg_augmentation = EEGAugmentation(
        noise_std=0.01,
        time_shift_range=0.05,
        amplitude_scale_range=(0.95, 1.05),
        channel_dropout_prob=0.05,
        time_mask_ratio=0.03,
    )

    # Data loading
    train_set = ThingsEEGDataset("train")
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=batch_size, shuffle=True
    )
    val_set = ThingsEEGDataset("val")
    val_loader = torch.utils.data.DataLoader(
        val_set, batch_size=batch_size, shuffle=False
    )

    Image_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print(f"Number of subjects: {train_set.num_subjects}")

    clip_model = EEGImageCLIP(
        seq_len=train_set.seq_len,
        in_channels=train_set.num_channels,
        embed_dim=256
    ).to("cuda")

    if clip_start_epoch < clip_epochs:

        # clip pre-training
        print("=" * 50)
        print("CLIP Pre-training")
        print("=" * 50)

        # load dataset
        image_dataset = ThingsImageDataset("extracted_training_images/training_images", transform=Image_transform)
        image_loader = DataLoader(image_dataset, batch_size=batch_size, shuffle=True)

        clip_model = pretrain_clip_model(
            clip_model,
            train_loader,
            image_loader,
            epochs=clip_epochs,
            lr=clip_lr,
            clip_start_epoch=clip_start_epoch
        )


    else:
        print("Loading pretrained model...")
        clip_model.load_state_dict(torch.load("clip_pretrained_model.pt"))

    model = EEGClassifier(
        clip_model,
        num_classes=train_set.num_classes,
        freeze_encoder=True
    ).to("cuda")

    train_augmented = AugmentedThingsEEGDataset(
        train_set, augmentation=eeg_augmentation, use_mixup=True
    )

    train_loader = torch.utils.data.DataLoader(
        train_augmented, batch_size=batch_size, shuffle=True
    )
    val_loader = torch.utils.data.DataLoader(
        val_set, batch_size=batch_size*2, shuffle=False
    )

    # Fine-tuning
    print("=" * 50)
    print("Step 2: Fine-tuning")
    print("=" * 50)

    optimizer = torch.optim.AdamW(
        model.classifier.parameters(),
        lr=finetune_lr * 5,
        weight_decay=0.05
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=finetune_lr * 10,
        epochs=finetune_epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.3
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # unfreeze schedule
    unfreeze_schedule = {
        'projection': 10,
        'conformer_blocks.3': 20,
        'conformer_blocks.2': 30,
        'conformer_blocks.1': 40,
        'conformer_blocks.0': 50,
    }

    max_val_acc = 0
    patience = 15
    patience_counter = 0

    if finetune_start_epoch >=1:
        checkpoint = torch.load("model_last.pt")
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        max_val_acc = checkpoint['max_val_acc']
        patience_counter = checkpoint['patience_counter']

    writer = SummaryWriter("tensorboard_pretrained")

    def accuracy(y_pred, y):
        return (y_pred.argmax(dim=-1) == y).float().mean()

    def apply_gradual_unfreezing(model, epoch):
        for layer_name, unfreeze_epoch in unfreeze_schedule.items():
            if epoch == unfreeze_epoch:
                print(f"Unfreezing {layer_name} at epoch {epoch}")
                params_to_add = []
                for name, param in model.eeg_encoder.named_parameters():
                    if layer_name in name:
                        param.requires_grad = True
                        params_to_add.append(param)
        if params_to_add:
            optimizer.add_param_group({'params': params_to_add, 'lr': finetune_lr})

    for epoch in range(finetune_start_epoch, finetune_epochs):
        print(f"Epoch {epoch+1}/{finetune_epochs}")

        apply_gradual_unfreezing(model, epoch)

        train_loss, train_acc, val_loss, val_acc = [], [], [], []

        # Training
        model.train()
        for batch in tqdm(train_loader, desc="Train"):
            if len(batch) == 3:
                X, y, subject_idxs = batch
                X, y = X.to("cuda"), y.to("cuda")

                y_pred = model(X)
                loss = criterion(y_pred, y)
            else:
                X, y_mixed, subject_idxs = batch
                X = X.to("cuda")
                y_mixed = (y_mixed[0].to("cuda"), y_mixed[1].to("cuda"), y_mixed[2])

                y_pred = model(X)
                loss = mixup_criterion(criterion, y_pred, y_mixed)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            train_loss.append(loss.item())
            if len(batch) == 3:
                train_acc.append(accuracy(y_pred, y).item())

        # Validation
        model.eval()
        with torch.no_grad():
            for X, y, subject_idxs in tqdm(val_loader, desc="Validation"):
                X, y = X.to("cuda"), y.to("cuda")

                y_pred = model(X)

                val_loss.append(F.cross_entropy(y_pred, y).item())
                val_acc.append(accuracy(y_pred, y).item())

        # Logging
        train_loss_avg = np.mean(train_loss)
        train_acc_avg = np.mean(train_acc) if train_acc else 0
        val_loss_avg = np.mean(val_loss)
        val_acc_avg = np.mean(val_acc)

        print(f"Epoch {epoch+1}/{finetune_epochs} | "
              f"train loss: {train_loss_avg:.3f} | "
              f"train acc: {train_acc_avg:.3f} | "
              f"val loss: {val_loss_avg:.3f} | "
              f"val acc: {val_acc_avg:.3f}")

        writer.add_scalar("train_loss", train_loss_avg, epoch)
        writer.add_scalar("train_acc", train_acc_avg, epoch)
        writer.add_scalar("val_loss", val_loss_avg, epoch)
        writer.add_scalar("val_acc", val_acc_avg, epoch)

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'max_val_acc': max_val_acc,
            'patience_counter': patience_counter,
        }, "model_last.pt")

        if val_acc_avg > max_val_acc:
            print("New best! Saving the model.")
            torch.save(model.state_dict(), "model_best.pt")
            max_val_acc = val_acc_avg
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered. ")
                break

    writer.close()
    return model

In [ ]:
model = train_with_pretraining(clip_start_epoch=30, finetune_start_epoch=0)

EEG: torch.Size([118800, 17, 100]), labels: torch.Size([118800]), subject indices: torch.Size([118800])
Number of subjects: 10
EEG: torch.Size([59400, 17, 100]), labels: torch.Size([59400]), subject indices: torch.Size([59400])
Number of subjects: 10
Number of subjects: 10


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 188MB/s]


Loading pretrained model...


/tmp/ipykernel_108/3893694592.py:56: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clip_model.load_state_dict(torch.load("clip_pretrained_model.pt"))


Step 2: Fine-tuning
Epoch 1/80


...

Validation:   0%|          | 0/465 [00:00<?, ?it/s]

Epoch 24/80 | train loss: 1.096 | train acc: 0.581 | val loss: 1.386 | val acc: 0.477
Epoch 25/80


Train:   0%|          | 0/929 [00:00<?, ?it/s]

Validation:   0%|          | 0/465 [00:00<?, ?it/s]

Epoch 25/80 | train loss: 1.089 | train acc: 0.584 | val loss: 1.375 | val acc: 0.481
Early stopping triggered. 


In [ ]:
# torch.cuda.memory_summary(device=None, abbreviated=False)

'|===========================================================================|\n|                  PyTorch CUDA memory summary, device ID 0                 |\n|---------------------------------------------------------------------------|\n|            CUDA OOMs: 3            |        cudaMalloc retries: 3         |\n|===========================================================================|\n|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |\n|---------------------------------------------------------------------------|\n| Allocated memory      |  22076 MiB |  22076 MiB |  32820 MiB |  10743 MiB |\n|       from large pool |  21947 MiB |  21948 MiB |  32657 MiB |  10710 MiB |\n|       from small pool |    129 MiB |    129 MiB |    162 MiB |     33 MiB |\n|---------------------------------------------------------------------------|\n| Active memory         |  22076 MiB |  22076 MiB |  32820 MiB |  10743 MiB |\n|       from large pool |  21947 MiB |  21948 MiB |

## 5.評価

In [ ]:
# ------------------
#    Dataloader
# ------------------
test_set = ThingsEEGDataset("test")
test_loader = torch.utils.data.DataLoader(
    test_set, batch_size=128, shuffle=False
)

# ------------------
#       Model
# ------------------
model.load_state_dict(torch.load("model_best.pt", map_location="cuda"))

# ------------------
#  Start evaluation
# ------------------
preds = []
model.eval()
for X, subject_idxs in tqdm(test_loader, desc="Evaluation"):
    preds.append(model(X.to("cuda")).detach().cpu())

preds = torch.cat(preds, dim=0).numpy()
np.save("submission", preds)
print(f"Submission {preds.shape} saved.")

EEG: torch.Size([59400, 17, 100]), labels: None, subject indices: torch.Size([59400])
Number of subjects: 10


/tmp/ipykernel_108/690991655.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("model_best.pt", map_location="cuda"))


Evaluation:   0%|          | 0/465 [00:00<?, ?it/s]

Submission (59400, 5) saved.


## 提出方法

以下の3点をzip化し，Omnicampusの「最終課題 (EEG)」から提出してください．

- `submission.npy`
- `model_last.pt`や`model_best.pt`など，テストに使用した重み（拡張子は`.pt`のみ）
- 本Colab Notebook

In [ ]:
from zipfile import ZipFile

model_path = "model_best.pt"
notebook_path = "DLBasics2025_competition_EEG_conformer_clip.ipynb"

with ZipFile("submission.zip", "w") as zf:
    zf.write("submission.npy")
    zf.write(model_path)
    zf.write(notebook_path)